# 01 Frame

Build the sampling frame: the threshold report's 115,396 acre assessment population (CWHR conifer outside urban and wilderness), minus edge buffers, steep ground, and existing plot footprints, with area accounting at every step. Output is one row per candidate 30 m pixel with forest type, LiDAR metrics, slope, road distance, state, ownership, and disturbance flags.

With `run.synthetic: true` a toy Basin is generated instead so the chain can be exercised. Replace the synthetic block with the real-layer block once layers are in `data/raw/`.

In [ ]:
import sys, os
print(sys.executable)
from pathlib import Path
os.chdir(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, str(Path.cwd()))
import numpy as np, pandas as pd
from src.io import load_config, get_logger
from src import strata, qa
cfg = load_config()
log = get_logger("01_frame")
log.info(f"Project: {cfg['project']['name']} | synthetic={cfg['run']['synthetic']} | freeze={cfg['run']['freeze']}")
P = Path(cfg["paths"]["processed"]); O = Path(cfg["paths"]["outputs"]); P.mkdir(parents=True, exist_ok=True); O.mkdir(exist_ok=True)

## Real layers (runs when `run.synthetic` is false)

Rasterize forest type to the LiDAR grid, sample the 2022 LiDAR metric rasters, DEM slope, and distance to roads at each pixel center, then apply exclusions. Every step logs acres removed.

In [ ]:
if not cfg["run"]["synthetic"]:
    import geopandas as gpd, rasterio
    from rasterio import features
    from shapely.geometry import Point
    from src.layers import read_layer, read_raster        # local file or TRPA REST service, per config.yaml
    src = cfg["sources"]; wcrs = cfg["crs"]["working"]; g = cfg["crs"]["lidar_grid_m"]

    # forest type from CWHR, mapped verbatim from threshold report Table 1
    veg = read_layer(src["cwhr_veg"], cfg, log=log)
    type_map = {c: t for t, cs in cfg["forest_types"].items() if isinstance(cs, list) for c in cs}
    veg["forest_type"] = veg["WHR_TYPE"].map(type_map)          # adjust field name to the layer
    veg = veg[veg["forest_type"].notna()]
    log.info(f"CWHR conifer polygons: {len(veg):,}; acres by type:\n{veg.assign(ac=veg.area/4046.86).groupby('forest_type')['ac'].sum().round()}")

    # exclusions
    urban = read_layer(src["urban_boundary"], cfg, log=log)
    wild = read_layer(src["wilderness"], cfg, log=log)
    pop = veg.overlay(pd.concat([urban[["geometry"]], wild[["geometry"]]]), how="difference")
    log.info(f"Population after urban + wilderness removal: {pop.area.sum()/4046.86:,.0f} ac (report: 115,396)")

    # grid of pixel centers on the LiDAR grid inside the population
    bbox = tuple(pop.total_bounds)
    p95, transform, rcrs = read_raster(src["p95_height_30m"], cfg, bbox=bbox, log=log)
    assert abs(transform.a - g) < 1e-6, f"expected {g} m grid, got {transform.a}"
    shape = p95.shape
    ft_raster = features.rasterize(((geom, i + 1) for i, geom in enumerate(pop.geometry)), out_shape=shape, transform=transform, fill=0)
    rows, cols = np.nonzero(ft_raster)
    xs, ys = rasterio.transform.xy(transform, rows, cols)
    frame = pd.DataFrame({"x": xs, "y": ys, "pixel_id": rows * shape[1] + cols})
    frame["forest_type"] = pop["forest_type"].values[ft_raster[rows, cols] - 1]
    frame["acres"] = g * g / 4046.86

    def sample(source, name):
        arr, tr, _ = read_raster(source, cfg, bbox=bbox, log=log)
        rr, cc = rasterio.transform.rowcol(tr, frame["x"].values, frame["y"].values)
        rr = np.clip(rr, 0, arr.shape[0] - 1); cc = np.clip(cc, 0, arr.shape[1] - 1)
        frame[name] = arr[rr, cc]
    sample(src["p95_height_30m"], "p95_height_m")
    sample(src["canopy_cover_30m"], "canopy_cover_pct")
    sample(src["stem_density_30m"], "stem_density")
    if Path(str(src.get("solid_frac_30m", ""))).exists() or str(src.get("solid_frac_30m", "")).startswith("http"):
        sample(src["solid_frac_30m"], "solid_frac")
    sample(src["dem"], "elev_m")   # slope/aspect: derive with rasterio or sample pre-computed rasters the same way

    pts = gpd.GeoDataFrame(frame, geometry=[Point(xy) for xy in zip(frame.x, frame.y)], crs=wcrs)
    edges = pd.concat([read_layer(src[k], cfg, log=log)[["geometry"]] for k in ["roads", "trails", "streams", "structures"]])
    buf = edges.buffer(cfg["frame"]["edge_buffer_m"]).union_all()
    pts["near_edge"] = pts.intersects(buf)
    roads = read_layer(src["roads"], cfg, log=log)
    pts["dist_road_m"] = pts.geometry.apply(lambda p: roads.distance(p).min())
    pts = pts.sjoin(read_layer(src["state_line"], cfg)[["STATE", "geometry"]], how="left").rename(columns={"STATE": "state"}).drop(columns="index_right")
    pts = pts.sjoin(read_layer(src["ownership"], cfg)[["OWNER", "geometry"]], how="left").rename(columns={"OWNER": "owner"}).drop(columns="index_right")
    fire = read_layer(src["fire_severity"], cfg, log=log); trt = read_layer(src["treatments_2027_2031"], cfg, log=log)
    pts["post_fire"] = pts.intersects(fire.union_all()); pts["treatment_2027_2031"] = pts.intersects(trt.union_all())
    # slope placeholder if no slope raster: compute from DEM outside this notebook and sample here
    if "slope_pct" not in pts: pts["slope_pct"] = np.nan
    frame = pd.DataFrame(pts.drop(columns="geometry"))
    frame = frame[~frame["near_edge"]]

## Synthetic Basin (runs when `run.synthetic` is true)

In [ ]:
if cfg["run"]["synthetic"]:
    frame = strata.make_synthetic_frame(cfg)
    log.info("Synthetic frame generated")
frame.head()

## Exclusions and area accounting

In [ ]:
acct = [{"step": "population", "acres": frame["acres"].sum(), "pixels": len(frame)}]
steep = frame["slope_pct"] > cfg["frame"]["max_slope_pct"]
acct.append({"step": f"removed slope > {cfg['frame']['max_slope_pct']}%", "acres": frame.loc[steep, "acres"].sum(), "pixels": int(steep.sum())})
frame = frame[~steep].copy()
if cfg["frame"]["max_access_distance_m"]:
    far = frame["dist_road_m"] > cfg["frame"]["max_access_distance_m"]
    acct.append({"step": "removed beyond access distance", "acres": frame.loc[far, "acres"].sum(), "pixels": int(far.sum())})
    frame = frame[~far].copy()
if "solid_frac" in frame and cfg["frame"].get("max_solid_fraction"):
    solid = (frame["solid_frac"] > cfg["frame"]["max_solid_fraction"]) & (frame["canopy_cover_pct"] > cfg["threshold"]["cover_sparse_pct"])
    acct.append({"step": f"removed solid-surface cells (single-return fraction > {cfg['frame']['max_solid_fraction']})", "acres": frame.loc[solid, "acres"].sum(), "pixels": int(solid.sum())})
    frame = frame[~solid].copy()
acct.append({"step": "frame", "acres": frame["acres"].sum(), "pixels": len(frame)})
acct = pd.DataFrame(acct)
for r in acct.itertuples(): log.info(f"{r.step}: {r.acres:,.0f} ac, {r.pixels:,} pixels")
acct.to_csv(O / "frame_accounting.csv", index=False)
frame["access_class"] = strata.access_class(frame["dist_road_m"], frame["slope_pct"], cfg)
frame.to_parquet(P / "frame.parquet", index=False)
log.info(f"Wrote frame: {len(frame):,} pixels, {frame['acres'].sum():,.0f} ac")
frame.groupby("forest_type")["acres"].sum().round()